# Compare Step 2 Infill Models

This notebook compares the original paper-style `AudioMotionTransformer` mask infiller against our `AudioFIMCausalLM` on the same classic Step 2 windows:

- left token frame `t`
- predict middle token frames `t+1, t+2, t+3`
- right token frame `t+4`
- HuBERT continuous audio features sampled onto the motion-token frame timeline

Metrics included:

- token accuracy, exact frame accuracy, exact gap accuracy, per-quantizer accuracy
- decoded RVQVAE body-feature MAE/MSE for paired debugging
- paper-style full-sequence evaluator FID/diversity in the later section

The first half is for debugging. Use the full-sequence evaluator section for reportable FID/diversity.

In [ ]:
from pathlib import Path
import json
import math
import os
import sys
from typing import Dict, List, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

def find_project_root(start: Path = Path.cwd()) -> Path:
    start = start.resolve()
    for path in [start, *start.parents]:
        if (path / "motion_generation").exists() and (path / "checkpoints").exists():
            return path
    raise RuntimeError("Could not find project root containing motion_generation/ and checkpoints/.")

PROJECT_ROOT = find_project_root()
MOTION_GENERATION_DIR = PROJECT_ROOT / "motion_generation"
EVALUATION_DIR = PROJECT_ROOT / "evaluation"
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(MOTION_GENERATION_DIR))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("project:", PROJECT_ROOT)
print("device:", DEVICE)

## Configuration

Keep `HISTORY_FRAMES_FOR_CAUSAL = 0` for the most apples-to-apples comparison, because the original mask transformer only sees left/right boundary frames. Raise it only if you intentionally want to test the causal model with extra history context.

In [ ]:
DATA_DIR = PROJECT_ROOT / "SuSuInterActs" / "SuSuInterActs"
MOTION_TOKEN_DIR = DATA_DIR / "motion_token_data"
EVAL_SPLIT_FILE = DATA_DIR / "split" / "val_file_list.txt"
MOTION2TEXT_JSON = DATA_DIR / "text_data" / "train.json"

MASK_CKPT = PROJECT_ROOT / "checkpoints" / "mask_transformer"
CAUSAL_CKPT = PROJECT_ROOT / "checkpoints" / "audio_fim_causal"

RVQVAE_CKPT = PROJECT_ROOT / "checkpoints" / "rvqvae" / "model" / "epoch_30.pth"
RVQVAE_MEAN_PATH = MOTION_GENERATION_DIR / "meta" / "mta_gen_demo" / "mean.npy"
RVQVAE_STD_PATH = MOTION_GENERATION_DIR / "meta" / "mta_gen_demo" / "std.npy"

# Source HuBERT feature fps and raw motion fps.
AUDIO_FPS = 50.0
RAW_MOTION_FPS = 20.0

# RVQVAE token frames are usually half the raw motion frame rate:
# raw 20 fps motion -> token 10 fps when unit_length=2.
MOTION_TOKEN_UNIT_LENGTH = 2.0
MOTION_TOKEN_FPS = RAW_MOTION_FPS / MOTION_TOKEN_UNIT_LENGTH

STEP = 4
HISTORY_FRAMES_FOR_CAUSAL = 0

# Increase these when the notebook flow is confirmed. Use None for full eval.
MAX_SEQUENCES = 128
MAX_WINDOWS = 512

# Old model masked-token generation. 1 is fastest; larger values fill masks in stages.
MASK_GENERATE_STEPS = 1

# Turn off if you only want token metrics or if RVQVAE files are unavailable.
DECODE_RVQVAE = True

# If True, clips invalid generated token IDs into 0..511 for RVQVAE decoding.
# Token metrics still count invalid IDs as wrong.
CLIP_INVALID_TOKENS_FOR_DECODE = True

AUDIO_FEAT_DIR = DATA_DIR / f"audio_features_hubert_layer9_fps{int(AUDIO_FPS)}"

for label, path in {
    "data": DATA_DIR,
    "motion tokens": MOTION_TOKEN_DIR,
    "audio features": AUDIO_FEAT_DIR,
    "eval split": EVAL_SPLIT_FILE,
    "mask checkpoint": MASK_CKPT,
    "causal checkpoint": CAUSAL_CKPT,
}.items():
    print(f"{label:18s}: {path} | exists={path.exists()}")

In [ ]:
import importlib

from safetensors import safe_open

from models.audio_motion_model import AudioMotionConfig, AudioMotionTransformer
from models.audio_fim_causal_model import AudioFIMCausalDataset, AudioFIMCausalLM
from scripts.train_audio_fim_causal import (
    discover_names,
    load_action_text_map,
    load_rvqvae_model_for_eval,
    load_sequences,
    motion_token_accuracy,
    read_split_file,
    decode_body_tokens_to_features,
)

def load_official_evaluator_helpers():
    """Load helpers from evaluation/evaluate_pred_motion_v2.py without
    leaving evaluation/ ahead of motion_generation/ on sys.path.
    """
    helper_names = [
        "calculate_alignment_single",
        "calculate_esd",
        "compute_fid_diversity_metrics",
        "extract_audio_beats",
        "extract_motion_beats",
        "extract_motion_beats_for_esd",
    ]
    saved_path = list(sys.path)
    conflict_prefixes = ("models", "datasets", "evaluate_pred_motion_v2")
    saved_modules = {
        name: module
        for name, module in list(sys.modules.items())
        if name in conflict_prefixes or name.startswith("models.") or name.startswith("datasets.")
    }
    for name in list(saved_modules):
        sys.modules.pop(name, None)
    sys.path.insert(0, str(EVALUATION_DIR))
    try:
        module = importlib.import_module("evaluate_pred_motion_v2")
        return {name: getattr(module, name) for name in helper_names}
    finally:
        sys.path[:] = saved_path
        for name in list(sys.modules):
            if name in conflict_prefixes or name.startswith("models.") or name.startswith("datasets."):
                sys.modules.pop(name, None)
        sys.modules.update(saved_modules)

_eval_helpers = load_official_evaluator_helpers()
calculate_alignment_single = _eval_helpers["calculate_alignment_single"]
calculate_esd = _eval_helpers["calculate_esd"]
compute_fid_diversity_metrics = _eval_helpers["compute_fid_diversity_metrics"]
extract_audio_beats = _eval_helpers["extract_audio_beats"]
extract_motion_beats = _eval_helpers["extract_motion_beats"]
extract_motion_beats_for_esd = _eval_helpers["extract_motion_beats_for_esd"]

def load_mask_transformer_local(ckpt_path: Path, device: torch.device) -> AudioMotionTransformer:
    config = AudioMotionConfig.from_pretrained(str(ckpt_path))
    model = AudioMotionTransformer(config)
    safetensors_path = ckpt_path / "model.safetensors"
    bin_path = ckpt_path / "pytorch_model.bin"
    if safetensors_path.exists():
        state_dict = {}
        with safe_open(str(safetensors_path), framework="pt", device="cpu") as f:
            for key in f.keys():
                state_dict[key] = f.get_tensor(key)
    elif bin_path.exists():
        state_dict = torch.load(bin_path, map_location="cpu")
    else:
        raise FileNotFoundError(f"No model.safetensors or pytorch_model.bin in {ckpt_path}")
    model.load_state_dict(state_dict, strict=True)
    return model.to(device).eval()

def require_path(path: Path, label: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {label}: {path}")

## Load Eval Windows

The dataset object is reused for window construction. It samples HuBERT features at `token_frame_idx * AUDIO_FPS / MOTION_TOKEN_FPS`, so both models receive audio aligned to token frames.

In [ ]:
require_path(MOTION_TOKEN_DIR, "motion token dir")
require_path(AUDIO_FEAT_DIR, "audio feature dir")
require_path(EVAL_SPLIT_FILE, "eval split file")

action_text_map = load_action_text_map(str(MOTION2TEXT_JSON) if MOTION2TEXT_JSON.exists() else None)
split_names = read_split_file(str(EVAL_SPLIT_FILE))
eval_names = discover_names(MOTION_TOKEN_DIR, AUDIO_FEAT_DIR, split_names)

eval_sequences = load_sequences(
    eval_names,
    MOTION_TOKEN_DIR,
    AUDIO_FEAT_DIR,
    action_text_map,
    max_samples=MAX_SEQUENCES,
    audio_fps=AUDIO_FPS,
    motion_fps=RAW_MOTION_FPS,
    motion_token_fps=MOTION_TOKEN_FPS,
    motion_token_unit_length=MOTION_TOKEN_UNIT_LENGTH,
)

eval_dataset = AudioFIMCausalDataset(
    eval_sequences,
    step=STEP,
    audio_fps=AUDIO_FPS,
    motion_fps=MOTION_TOKEN_FPS,
    min_history_frames=HISTORY_FRAMES_FOR_CAUSAL,
    max_history_frames=HISTORY_FRAMES_FOR_CAUSAL,
    max_windows_per_sequence=None,
    seed=42,
)

num_windows = len(eval_dataset) if MAX_WINDOWS is None else min(int(MAX_WINDOWS), len(eval_dataset))
window_indices = np.linspace(0, len(eval_dataset) - 1, num_windows, dtype=np.int64).tolist()

print("sequences:", len(eval_sequences))
print("windows:", len(eval_dataset))
print("selected windows:", len(window_indices))
print("motion token fps:", MOTION_TOKEN_FPS)
print("audio fps:", AUDIO_FPS)

## Load Models

In [ ]:
require_path(MASK_CKPT, "mask transformer checkpoint")
require_path(CAUSAL_CKPT, "causal AudioFIM checkpoint")

mask_model = load_mask_transformer_local(MASK_CKPT, DEVICE)
causal_model = AudioFIMCausalLM.from_pretrained(str(CAUSAL_CKPT)).to(DEVICE).eval()

print("mask params:", sum(p.numel() for p in mask_model.parameters()))
print("causal params:", sum(p.numel() for p in causal_model.parameters()))

## Prediction Adapters

`mask_model` predicts all masked middle positions from `[left] [MASK x 12] [right]`.

`causal_model` sees its compact FIM prompt and autoregressively generates the middle tokens after the `[MIDDLE_MOTION]` marker.

In [ ]:
def tensor_to_numpy(x):
    if x is None:
        return None
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)

def audio5_from_example(example) -> np.ndarray:
    parts = [
        tensor_to_numpy(example.left_audio_feature),
        tensor_to_numpy(example.middle_audio_features),
        tensor_to_numpy(example.right_audio_feature),
    ]
    return np.concatenate([parts[0][None, :], parts[1], parts[2][None, :]], axis=0).astype(np.float32)

@torch.no_grad()
def predict_mask_middle(example) -> List[List[int]]:
    audio5 = audio5_from_example(example)
    ntpf = mask_model.config.num_tokens_per_frame
    codebook_size = mask_model.config.codebook_size
    offsets = [codebook_size * i for i in range(ntpf)]
    mask_token_id = mask_model.config.vocab_size - 1

    input_tokens = []
    for q in range(ntpf):
        input_tokens.append(int(example.left_anchor[q]) + offsets[q])
    input_tokens.extend([mask_token_id] * (3 * ntpf))
    for q in range(ntpf):
        input_tokens.append(int(example.right_anchor[q]) + offsets[q])

    input_ids = torch.tensor([input_tokens], dtype=torch.long, device=DEVICE)
    audio_feat = torch.tensor(audio5, dtype=torch.float32, device=DEVICE).unsqueeze(0)
    output = mask_model.generate_sbs(input_ids, audio_feat, generate_steps=MASK_GENERATE_STEPS)
    output = output[0].detach().cpu().tolist()

    frames = []
    for frame_idx in range(1, 4):
        frame = []
        for q in range(ntpf):
            token_id = int(output[frame_idx * ntpf + q])
            frame.append(token_id - offsets[q])
        frames.append(frame)
    return frames

@torch.no_grad()
def predict_causal_middle(example) -> List[List[int]]:
    pred = causal_model.generate_infill(
        history_motion=example.history_motion,
        left_anchor=example.left_anchor,
        right_anchor=example.right_anchor,
        middle_audio_features=example.middle_audio_features,
        left_audio_feature=example.left_audio_feature,
        right_audio_feature=example.right_audio_feature,
        history_audio_features=example.history_audio_features,
        temperature=0.0,
    )
    return [list(map(int, frame)) for frame in pred]

def sanitize_tokens_for_decode(frames: Sequence[Sequence[int]], codebook_size: int = 512) -> List[List[int]]:
    arr = np.asarray(frames, dtype=np.int64)
    if CLIP_INVALID_TOKENS_FOR_DECODE:
        arr = np.clip(arr, 0, codebook_size - 1)
    return arr.tolist()

def invalid_token_fraction(frames: Sequence[Sequence[int]], codebook_size: int = 512) -> float:
    arr = np.asarray(frames, dtype=np.int64)
    return float(((arr < 0) | (arr >= codebook_size)).mean()) if arr.size else 0.0

## Metric Helpers

In [ ]:
def token_metrics(gt: Sequence[Sequence[int]], pred: Sequence[Sequence[int]]) -> Dict[str, float]:
    gt_arr = np.asarray(gt, dtype=np.int64)
    pred_arr = np.asarray(pred, dtype=np.int64)
    if gt_arr.shape != pred_arr.shape:
        return {"token_acc": np.nan, "exact_frame_acc": np.nan, "exact_gap_acc": 0.0}
    matches = gt_arr == pred_arr
    metrics = {
        "token_acc": float(matches.mean()),
        "exact_frame_acc": float(matches.all(axis=1).mean()),
        "exact_gap_acc": float(matches.all()),
    }
    for q in range(gt_arr.shape[1]):
        metrics[f"q{q + 1}_acc"] = float(matches[:, q].mean())
    return metrics

def decoded_feature_metrics(gt_feat: np.ndarray, pred_feat: np.ndarray) -> Dict[str, float]:
    diff = pred_feat.astype(np.float64) - gt_feat.astype(np.float64)
    vel_gt = np.diff(gt_feat, axis=0)
    vel_pred = np.diff(pred_feat, axis=0)
    vel_diff = vel_pred.astype(np.float64) - vel_gt.astype(np.float64)
    out = {
        "body_feat_mae": float(np.mean(np.abs(diff))),
        "body_feat_mse": float(np.mean(diff ** 2)),
        "body_feat_rmse": float(np.sqrt(np.mean(diff ** 2))),
    }
    if len(vel_diff) > 0:
        out["body_velocity_mse"] = float(np.mean(vel_diff ** 2))
    else:
        out["body_velocity_mse"] = np.nan
    return out

## Optional RVQVAE Decoder

This decodes RVQ token frames into normalized body feature vectors. The Frechet/FID-style metric below is computed in this decoded feature space.

In [ ]:
rvqvae = None
rvq_mean = None
rvq_std = None

if DECODE_RVQVAE:
    require_path(RVQVAE_CKPT, "RVQVAE checkpoint")
    require_path(RVQVAE_MEAN_PATH, "RVQVAE mean")
    require_path(RVQVAE_STD_PATH, "RVQVAE std")
    rvqvae, rvq_config = load_rvqvae_model_for_eval(RVQVAE_CKPT, DEVICE)
    rvq_mean = torch.tensor(np.load(RVQVAE_MEAN_PATH), dtype=torch.float32, device=DEVICE)
    rvq_std = torch.tensor(np.load(RVQVAE_STD_PATH), dtype=torch.float32, device=DEVICE)
    print("loaded RVQVAE:", RVQVAE_CKPT)
else:
    print("RVQVAE decoding disabled.")

@torch.no_grad()
def decode_body_features(frames: Sequence[Sequence[int]]) -> np.ndarray:
    if rvqvae is None:
        raise RuntimeError("RVQVAE decoding is disabled.")
    clean = sanitize_tokens_for_decode(frames)
    feat = decode_body_tokens_to_features(rvqvae, clean, rvq_mean, rvq_std, DEVICE)
    return feat.detach().cpu().numpy().astype(np.float32)

## Run Comparison

In [ ]:
rows = []

for idx in tqdm(window_indices, desc="compare windows"):
    example = eval_dataset[int(idx)]
    gt_middle = [list(map(int, frame)) for frame in example.middle_motion]
    predictions = {
        "mask": predict_mask_middle(example),
        "causal": predict_causal_middle(example),
    }

    decoded_gt_middle = None
    if DECODE_RVQVAE:
        decoded_gt_middle = decode_body_features(gt_middle)

    for model_name, pred_middle in predictions.items():
        row = {
            "model": model_name,
            "dataset_idx": int(idx),
            "name": example.name,
            "left_idx": int(example.left_idx),
            "right_idx": int(example.right_idx),
            "invalid_token_frac": invalid_token_fraction(pred_middle),
        }
        row.update(token_metrics(gt_middle, pred_middle))

        if DECODE_RVQVAE:
            pred_middle_clean = sanitize_tokens_for_decode(pred_middle)
            decoded_pred_middle = decode_body_features(pred_middle_clean)
            row.update(decoded_feature_metrics(decoded_gt_middle, decoded_pred_middle))

        rows.append(row)

results_df = pd.DataFrame(rows)
results_df.head()

## Aggregate Results

In [ ]:
mean_cols = [col for col in results_df.columns if col not in {"model", "name"}]
summary = results_df.groupby("model")[mean_cols].mean(numeric_only=True).T
display(summary)

## Inspect Best/Worst Windows

This is useful when the average metrics look okay but free-running examples still look wrong.

In [ ]:
sort_col = "token_acc"
for model_name in sorted(results_df["model"].unique()):
    print("\n", "=" * 80)
    print(model_name, "best")
    display(results_df[results_df.model == model_name].sort_values(sort_col, ascending=False).head(5))
    print(model_name, "worst")
    display(results_df[results_df.model == model_name].sort_values(sort_col, ascending=True).head(5))

## Save Metric Tables

## Paper-Style Full-Sequence Evaluator

The window-level FID above is only a lightweight diagnostic. This section builds full reconstructed clips and evaluates them with the repository's evaluator motion encoder from `checkpoints/eval_model/best_model.pt`.

Set `RUN_SEQUENCE_EXPORT = True` first to write full-sequence `.npy` files for GT, `codec_gt`, mask, and causal. Then set `RUN_EVALUATOR_FID = True` to compute evaluator-latent FID/diversity. The output format is compatible with `evaluation/datasets/pred_motion_dataset.py`.

`codec_gt` is decoded ground-truth RVQ tokens compared against the selected GT reference. When `OFFICIAL_GT_SOURCE = "raw_motion_data"`, `codec_gt` estimates the codec/tokenization/eval-prep floor before any Step 2 infill error is added.

In [ ]:
RUN_SEQUENCE_EXPORT = False
RUN_EVALUATOR_FID = False
RUN_BEAT_METRICS = False

# Use None for all eval sequences. Start with a small number while checking speed.
OFFICIAL_MAX_SEQUENCES = 128

# raw_motion_data is closer to final-motion evaluation; decoded_tokens is a cleaner codec-only baseline.
OFFICIAL_GT_SOURCE = "raw_motion_data"  # "raw_motion_data" or "decoded_tokens"

OFFICIAL_OUT_ROOT = MOTION_GENERATION_DIR / "notebooks" / "step2_official_eval_outputs"
OFFICIAL_GT_DIR = OFFICIAL_OUT_ROOT / "gt"
OFFICIAL_CODEC_GT_DIR = OFFICIAL_OUT_ROOT / "codec_gt"
OFFICIAL_MASK_DIR = OFFICIAL_OUT_ROOT / "mask"
OFFICIAL_CAUSAL_DIR = OFFICIAL_OUT_ROOT / "causal"

EVALUATOR_CKPT = PROJECT_ROOT / "checkpoints" / "eval_model" / "best_model.pt"
EVALUATOR_CFG_PATH = PROJECT_ROOT / "evaluation" / "config" / "train_bert_orig.yaml"
EVALUATOR_STATS_DIR = PROJECT_ROOT / "evaluation" / "stats" / "humanml3d" / "guoh3dfeats"
MOTION_DATA_DIR = DATA_DIR / "motion_data"
WAV_DIR = DATA_DIR / "wav_data"

print("output root:", OFFICIAL_OUT_ROOT)
print("evaluator ckpt:", EVALUATOR_CKPT, EVALUATOR_CKPT.exists())
print("evaluator cfg:", EVALUATOR_CFG_PATH, EVALUATOR_CFG_PATH.exists())
print("evaluator stats:", EVALUATOR_STATS_DIR, EVALUATOR_STATS_DIR.exists())

In [ ]:
def audio_feature_for_token_frame(item: Dict, token_frame_idx: int) -> torch.Tensor:
    audio = item["audio_features"]
    audio_fps = float(item.get("audio_fps", AUDIO_FPS))
    token_fps = float(item.get("motion_fps", MOTION_TOKEN_FPS))
    audio_idx = int(round(float(token_frame_idx) * audio_fps / token_fps))
    audio_idx = max(0, min(audio_idx, len(audio) - 1))
    return torch.tensor(audio[audio_idx], dtype=torch.float32)

def audio5_for_token_window(item: Dict, left_idx: int) -> np.ndarray:
    frames = [audio_feature_for_token_frame(item, left_idx + i).numpy() for i in range(STEP + 1)]
    return np.stack(frames, axis=0).astype(np.float32)

@torch.no_grad()
def predict_mask_window_from_item(item: Dict, left_idx: int) -> List[List[int]]:
    tokens = item["motion_tokens"]
    ntpf = mask_model.config.num_tokens_per_frame
    codebook_size = mask_model.config.codebook_size
    offsets = [codebook_size * i for i in range(ntpf)]
    mask_token_id = mask_model.config.vocab_size - 1
    right_idx = left_idx + STEP

    input_tokens = []
    for q in range(ntpf):
        input_tokens.append(int(tokens[left_idx][q]) + offsets[q])
    input_tokens.extend([mask_token_id] * ((STEP - 1) * ntpf))
    for q in range(ntpf):
        input_tokens.append(int(tokens[right_idx][q]) + offsets[q])

    input_ids = torch.tensor([input_tokens], dtype=torch.long, device=DEVICE)
    audio_feat = torch.tensor(audio5_for_token_window(item, left_idx), dtype=torch.float32, device=DEVICE).unsqueeze(0)
    output = mask_model.generate_sbs(input_ids, audio_feat, generate_steps=MASK_GENERATE_STEPS)[0].detach().cpu().tolist()

    pred = []
    for frame_idx in range(1, STEP):
        frame = []
        for q in range(ntpf):
            frame.append(int(output[frame_idx * ntpf + q]) - offsets[q])
        pred.append(frame)
    return pred

@torch.no_grad()
def predict_causal_window_from_item(item: Dict, left_idx: int) -> List[List[int]]:
    tokens = item["motion_tokens"]
    right_idx = left_idx + STEP
    history_start = max(0, left_idx - HISTORY_FRAMES_FOR_CAUSAL)
    history_indices = list(range(history_start, left_idx))
    middle_indices = list(range(left_idx + 1, right_idx))

    history_audio = None
    if history_indices:
        history_audio = torch.stack([audio_feature_for_token_frame(item, i) for i in history_indices], dim=0)

    return causal_model.generate_infill(
        history_motion=[list(tokens[i]) for i in history_indices],
        left_anchor=list(tokens[left_idx]),
        right_anchor=list(tokens[right_idx]),
        middle_audio_features=torch.stack([audio_feature_for_token_frame(item, i) for i in middle_indices], dim=0),
        left_audio_feature=audio_feature_for_token_frame(item, left_idx),
        right_audio_feature=audio_feature_for_token_frame(item, right_idx),
        history_audio_features=history_audio,
        temperature=0.0,
    )

def build_full_token_sequence(item: Dict, model_name: str) -> Tuple[List[List[int]], List[List[int]]]:
    tokens = item["motion_tokens"]
    left_indices = list(range(0, len(tokens) - STEP, STEP))
    if not left_indices:
        return [], []

    pred_full = []
    gt_full = []
    for window_idx, left_idx in enumerate(left_indices):
        right_idx = left_idx + STEP
        if model_name == "mask":
            middle = predict_mask_window_from_item(item, left_idx)
        elif model_name == "causal":
            middle = predict_causal_window_from_item(item, left_idx)
        else:
            raise ValueError(f"unknown model_name={model_name}")

        if window_idx == 0:
            pred_full.append(list(tokens[left_idx]))
            gt_full.append(list(tokens[left_idx]))
        pred_full.extend([list(frame) for frame in middle])
        pred_full.append(list(tokens[right_idx]))
        gt_full.extend([list(tokens[i]) for i in range(left_idx + 1, right_idx + 1)])

    return gt_full, pred_full

def decode_tokens_to_evaluator_body(tokens: Sequence[Sequence[int]]) -> np.ndarray:
    # RVQVAE decoder already restores token frames to source motion frames
    # (for this checkpoint: 10fps tokens -> 20fps body features).
    return decode_body_features(sanitize_tokens_for_decode(tokens))

def load_raw_gt_body(name: str, target_len: int) -> np.ndarray | None:
    path = MOTION_DATA_DIR / f"{name}.npy"
    if not path.exists():
        return None
    raw = np.load(path, allow_pickle=True)
    if isinstance(raw, np.ndarray) and raw.dtype == object:
        raw = raw.item()
    if not isinstance(raw, dict) or "body" not in raw:
        return None
    body = np.asarray(raw["body"], dtype=np.float32)
    return body[:target_len]

def save_evaluator_motion(path: Path, name: str, body: np.ndarray) -> None:
    body = np.asarray(body, dtype=np.float32)
    zeros_hand = np.zeros((len(body), 120), dtype=np.float32)
    payload = {"name": name, "body": body, "left": zeros_hand, "right": zeros_hand}
    np.save(path, payload)

def clean_eval_dir(path: Path, suffix: str) -> None:
    path.mkdir(parents=True, exist_ok=True)
    for old_file in path.glob(f"*_{suffix}.npy"):
        old_file.unlink()

In [ ]:
def export_full_sequence_eval_npys() -> pd.DataFrame:
    if not DECODE_RVQVAE:
        raise RuntimeError("Set DECODE_RVQVAE = True before exporting evaluator npys.")
    require_path(RVQVAE_CKPT, "RVQVAE checkpoint")

    selected_sequences = load_sequences(
        eval_names,
        MOTION_TOKEN_DIR,
        AUDIO_FEAT_DIR,
        action_text_map,
        max_samples=OFFICIAL_MAX_SEQUENCES,
        audio_fps=AUDIO_FPS,
        motion_fps=RAW_MOTION_FPS,
        motion_token_fps=MOTION_TOKEN_FPS,
        motion_token_unit_length=MOTION_TOKEN_UNIT_LENGTH,
    )

    clean_eval_dir(OFFICIAL_GT_DIR, "gt")
    clean_eval_dir(OFFICIAL_CODEC_GT_DIR, "pred")
    clean_eval_dir(OFFICIAL_MASK_DIR, "pred")
    clean_eval_dir(OFFICIAL_CAUSAL_DIR, "pred")

    rows = []
    for seq_idx, item in enumerate(tqdm(selected_sequences, desc="export full sequences")):
        gt_tokens, mask_tokens = build_full_token_sequence(item, "mask")
        _, causal_tokens = build_full_token_sequence(item, "causal")
        if not gt_tokens or not mask_tokens or not causal_tokens:
            continue

        mask_body = decode_tokens_to_evaluator_body(mask_tokens)
        causal_body = decode_tokens_to_evaluator_body(causal_tokens)
        decoded_gt_body = decode_tokens_to_evaluator_body(gt_tokens)
        decoded_frames = len(decoded_gt_body)
        target_len = min(len(mask_body), len(causal_body), decoded_frames)
        raw_gt = load_raw_gt_body(item["name"], target_len)
        raw_frames = 0 if raw_gt is None else len(raw_gt)

        if OFFICIAL_GT_SOURCE == "raw_motion_data":
            gt_body = raw_gt if raw_gt is not None and len(raw_gt) > 0 else decoded_gt_body
        elif OFFICIAL_GT_SOURCE == "decoded_tokens":
            gt_body = decoded_gt_body
        else:
            raise ValueError("OFFICIAL_GT_SOURCE must be raw_motion_data or decoded_tokens")

        target_len = min(len(gt_body), len(mask_body), len(causal_body))
        if target_len < 2:
            continue
        gt_body = gt_body[:target_len]
        codec_gt_body = decoded_gt_body[:target_len]
        mask_body = mask_body[:target_len]
        causal_body = causal_body[:target_len]

        file_stem = f"{seq_idx:06d}"
        save_evaluator_motion(OFFICIAL_GT_DIR / f"{file_stem}_gt.npy", item["name"], gt_body)
        save_evaluator_motion(OFFICIAL_CODEC_GT_DIR / f"{file_stem}_pred.npy", item["name"], codec_gt_body)
        save_evaluator_motion(OFFICIAL_MASK_DIR / f"{file_stem}_pred.npy", item["name"], mask_body)
        save_evaluator_motion(OFFICIAL_CAUSAL_DIR / f"{file_stem}_pred.npy", item["name"], causal_body)

        rows.append({
            "file_stem": file_stem,
            "name": item["name"],
            "frames_20fps": target_len,
            "token_frames": len(gt_tokens),
            "decoded_frames": decoded_frames,
            "raw_frames_before_trim": raw_frames,
            "gt_source": OFFICIAL_GT_SOURCE,
        })

    manifest = pd.DataFrame(rows)
    OFFICIAL_OUT_ROOT.mkdir(parents=True, exist_ok=True)
    manifest.to_csv(OFFICIAL_OUT_ROOT / "manifest.csv", index=False)
    return manifest

if RUN_SEQUENCE_EXPORT:
    official_manifest = export_full_sequence_eval_npys()
    display(official_manifest.head())
    print("exported clips:", len(official_manifest))
else:
    print("Set RUN_SEQUENCE_EXPORT = True to export full-sequence evaluator npys.")

In [ ]:
from types import SimpleNamespace

def load_yaml_namespace(path: Path):
    import yaml
    with open(path, "r", encoding="utf-8") as f:
        data = yaml.safe_load(f)
    def convert(value):
        if isinstance(value, dict):
            return SimpleNamespace(**{k: convert(v) for k, v in value.items()})
        return value
    return convert(data)

def evaluator_length_to_mask(lengths: torch.Tensor, max_length: int, device: torch.device) -> torch.Tensor:
    lengths = lengths.to(device=device, dtype=torch.long)
    positions = torch.arange(max_length, device=device).unsqueeze(0)
    return positions < lengths.unsqueeze(1)

def load_evaluator_motion_encoder(device: torch.device):
    from evaluation.models.actor import ACTORStyleEncoder
    cfg = load_yaml_namespace(EVALUATOR_CFG_PATH)
    encoder = ACTORStyleEncoder(
        cfg.motion_encoder.nfeats,
        cfg.motion_encoder.vae,
        cfg.motion_encoder.latent_dim,
        cfg.motion_encoder.ff_size,
        cfg.motion_encoder.num_layers,
        cfg.motion_encoder.num_heads,
        cfg.motion_encoder.dropout,
        cfg.motion_encoder.activation,
    )
    ckpt = torch.load(EVALUATOR_CKPT, map_location="cpu")
    if isinstance(ckpt, dict) and "state_dict" in ckpt:
        ckpt = ckpt["state_dict"]
    motion_state = {
        key.replace("motion_encoder.", "", 1): value
        for key, value in ckpt.items()
        if key.startswith("motion_encoder.")
    }
    missing, unexpected = encoder.load_state_dict(motion_state, strict=False)
    if missing or unexpected:
        print("motion encoder missing keys:", missing)
        print("motion encoder unexpected keys:", unexpected)
    encoder.to(device).eval()
    return cfg, encoder

def load_evaluator_body(path: Path) -> Tuple[str, np.ndarray]:
    raw = np.load(path, allow_pickle=True)
    if isinstance(raw, np.ndarray) and raw.shape == ():
        raw = raw.item()
    if isinstance(raw, dict):
        return raw.get("name", path.stem), np.asarray(raw["body"], dtype=np.float32)
    return path.stem, np.asarray(raw, dtype=np.float32)

def normalize_pad_body_for_evaluator(body: np.ndarray, mean: torch.Tensor, std: torch.Tensor, max_len: int) -> Tuple[torch.Tensor, int]:
    body = np.asarray(body, dtype=np.float32)
    length = min(len(body), max_len)
    body = body[:length]
    body_tensor = torch.tensor(body, dtype=torch.float32)
    body_dim = body_tensor.shape[1]
    body_tensor = (body_tensor - mean[:body_dim]) / (std[:body_dim] + 1e-12)
    if length < max_len:
        pad = torch.zeros((max_len - length, body_dim), dtype=torch.float32)
        body_tensor = torch.cat([body_tensor, pad], dim=0)
    return body_tensor, length

@torch.no_grad()
def encode_evaluator_latents(motion_dir: Path, suffix: str, cfg, encoder, device: torch.device, batch_size: int = 64) -> Tuple[List[str], np.ndarray]:
    files = sorted(motion_dir.glob(f"*_{suffix}.npy"))
    if not files:
        raise FileNotFoundError(f"No *_{suffix}.npy files found in {motion_dir}")
    mean = torch.load(EVALUATOR_STATS_DIR / "mean.pt", map_location="cpu").float()
    std = torch.load(EVALUATOR_STATS_DIR / "std.pt", map_location="cpu").float()
    max_len = int(cfg.dataset.max_motion_length)

    names = []
    latents = []
    batch_motions = []
    batch_lengths = []
    for path in tqdm(files, desc=f"encode {motion_dir.name}"):
        name, body = load_evaluator_body(path)
        motion, length = normalize_pad_body_for_evaluator(body, mean, std, max_len)
        names.append(name)
        batch_motions.append(motion)
        batch_lengths.append(length)
        if len(batch_motions) >= batch_size:
            x = torch.stack(batch_motions, dim=0).to(device)
            lengths = torch.tensor(batch_lengths, dtype=torch.long, device=device)
            mask = evaluator_length_to_mask(lengths, max_len, device)
            encoded = encoder({"x": x, "mask": mask})[:, 0]
            latents.append(encoded.detach().cpu())
            batch_motions.clear()
            batch_lengths.clear()
    if batch_motions:
        x = torch.stack(batch_motions, dim=0).to(device)
        lengths = torch.tensor(batch_lengths, dtype=torch.long, device=device)
        mask = evaluator_length_to_mask(lengths, max_len, device)
        encoded = encoder({"x": x, "mask": mask})[:, 0]
        latents.append(encoded.detach().cpu())
    return names, torch.cat(latents, dim=0).numpy()

# FID/Diversity below intentionally calls compute_fid_diversity_metrics()
# imported from evaluation/evaluate_pred_motion_v2.py.

In [ ]:
if RUN_EVALUATOR_FID:
    require_path(EVALUATOR_CKPT, "evaluator checkpoint")
    require_path(EVALUATOR_CFG_PATH, "evaluator config")
    cfg_eval, motion_encoder = load_evaluator_motion_encoder(DEVICE)
    _, gt_latents = encode_evaluator_latents(OFFICIAL_GT_DIR, "gt", cfg_eval, motion_encoder, DEVICE)
    evaluator_rows = []
    for model_name, motion_dir in {
        "codec_gt": OFFICIAL_CODEC_GT_DIR,
        "mask": OFFICIAL_MASK_DIR,
        "causal": OFFICIAL_CAUSAL_DIR,
    }.items():
        _, pred_latents = encode_evaluator_latents(motion_dir, "pred", cfg_eval, motion_encoder, DEVICE)
        metrics = compute_fid_diversity_metrics(gt_latents, pred_latents, diversity_times=300)
        metrics = {key: round(float(value), 6) for key, value in metrics.items()}
        metrics["model"] = model_name
        metrics["num_clips"] = int(len(pred_latents))
        evaluator_rows.append(metrics)
    evaluator_df = pd.DataFrame(evaluator_rows).set_index("model")
    display(evaluator_df)
else:
    print("Set RUN_EVALUATOR_FID = True after exporting full-sequence npys.")

In [ ]:
def wav_path_for_name(name: str) -> Path:
    return WAV_DIR / f"{name}.wav"

def compute_exported_beat_metrics(motion_dir: Path, suffix: str = "pred", fps: int = 20, tolerance: float = 0.1) -> Dict[str, float]:
    bas_distances, bhr_scores, esd_scores = [], [], []
    used = 0
    for path in tqdm(sorted(motion_dir.glob(f"*_{suffix}.npy")), desc=f"beat {motion_dir.name}"):
        name, body = load_evaluator_body(path)
        wav_path = wav_path_for_name(name)
        if not wav_path.exists():
            continue
        audio_beats = extract_audio_beats(str(wav_path))
        motion_beats = extract_motion_beats(body, fps=fps)
        bas, bhr = calculate_alignment_single(motion_beats, audio_beats, tolerance=tolerance)
        esd = calculate_esd(audio_beats, extract_motion_beats_for_esd(body, fps=fps))
        if bas is not None and not np.isnan(bas):
            bas_distances.append(bas)
        if bhr is not None:
            bhr_scores.append(bhr)
        esd_scores.append(esd)
        used += 1
    return {
        "BAS_distance_lower_better": float(np.mean(bas_distances)) if bas_distances else float("nan"),
        "BHR_higher_better": float(np.mean(bhr_scores)) if bhr_scores else float("nan"),
        "ESD_lower_better": float(np.mean(esd_scores)) if esd_scores else float("nan"),
        "num_evaluated": used,
    }

if RUN_BEAT_METRICS:
    beat_rows = []
    for model_name, motion_dir, suffix in [
        ("gt", OFFICIAL_GT_DIR, "gt"),
        ("codec_gt", OFFICIAL_CODEC_GT_DIR, "pred"),
        ("mask", OFFICIAL_MASK_DIR, "pred"),
        ("causal", OFFICIAL_CAUSAL_DIR, "pred"),
    ]:
        metrics = compute_exported_beat_metrics(motion_dir, suffix=suffix, fps=20, tolerance=0.1)
        metrics["model"] = model_name
        beat_rows.append(metrics)
    beat_df = pd.DataFrame(beat_rows).set_index("model")
    display(beat_df)
else:
    print("Set RUN_BEAT_METRICS = True after exporting full-sequence npys.")

In [ ]:
SAVE_RESULTS = False
OUT_DIR = MOTION_GENERATION_DIR / "notebooks" / "step2_metric_outputs"

if SAVE_RESULTS:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    results_df.to_csv(OUT_DIR / "window_metrics.csv", index=False)
    summary.to_csv(OUT_DIR / "summary_metrics.csv")
    print("saved to", OUT_DIR)
else:
    print("Set SAVE_RESULTS = True to write CSV outputs.")